# Azzaro: Whisper turbo vs small vs large

Comparamos tres modelos de Whisper sobre el mismo video, contamos diferencias de palabras y revisamos los clips donde no coinciden. El caso puntual a mirar es `racingista` / `racinguista`.

## 1. Configuracion

In [1]:
from itertools import combinations
from pathlib import Path
import html
import os
import sys
from urllib.parse import quote

ROOT = Path.cwd().resolve()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import HTML, Markdown, display

from evaluation.src.whisper_model_comparison import (
    alinear_diferencias,
    descargar_video_yt,
    exportar_clips_diferencias,
    extraer_palabras,
    normalizar_texto,
    transcribir_whisper,
)

VIDEO_URL = "https://www.youtube.com/watch?v=a4ggqJZXnQE"
VIDEO_PATH = None  # si ya tenes el mp4 local, poner aca la ruta y dejar VIDEO_URL como esta

MODELOS = ["turbo", "small", "large"]
COOKIES_FROM_BROWSER = "chrome"  # poner None si YouTube no pide login
FORCE_TRANSCRIBE = False

CLIP_MARGIN_SECONDS = 0.0  # 0.0 muestra el intervalo real usado por la transcripcion original
MAX_CASOS_A_MOSTRAR = None  # poner un numero si queres limitar la revision visual

NOTEBOOK_DIR = ROOT / "evaluation" / "notebooks"
WORKDIR = ROOT / "evaluation" / "outputs" / "azzaro_whisper"
WORKDIR


WindowsPath('C:/Users/bianc/NLP Natural Language Processing/labios-argentos/evaluation/outputs/azzaro_whisper')

## 2. Descargar o cargar video

In [2]:
if VIDEO_PATH:
    video_path = Path(VIDEO_PATH).expanduser().resolve()
else:
    video_path = descargar_video_yt(
        VIDEO_URL,
        WORKDIR / "videos",
        nombre_base="azzaro_racing_caracas",
        cookies_from_browser=COOKIES_FROM_BROWSER,
    )

video_path

WindowsPath('C:/Users/bianc/NLP Natural Language Processing/labios-argentos/evaluation/outputs/azzaro_whisper/videos/azzaro_racing_caracas.mp4')

## 3. Transcribir con turbo, small y large

Esto cachea cada JSON en `evaluation/outputs/azzaro_whisper/transcripts/`. Si ya existe, no vuelve a transcribir.

In [3]:
resultados = {}
palabras = {}

for model_name in MODELOS:
    print(f"Transcribiendo / cargando cache: {model_name}")
    resultados[model_name] = transcribir_whisper(
        video_path,
        model_name=model_name,
        output_dir=WORKDIR / "transcripts",
        force=FORCE_TRANSCRIBE,
    )
    palabras[model_name] = extraer_palabras(resultados[model_name], model_name)

pd.DataFrame([
    {"modelo": modelo, "palabras": len(palabras_modelo)}
    for modelo, palabras_modelo in palabras.items()
])

Transcribiendo / cargando cache: turbo
Transcribiendo / cargando cache: small
Transcribiendo / cargando cache: large


,modelo,palabras
0,turbo,2611
1,small,2590
2,large,2684


## 4. Detectar diferencias donde difieren los tres modelos

A partir de aca filtramos fuerte: solo quedan tramos donde `turbo`, `small` y `large` producen tres textos distintos.


In [4]:
def texto_modelo_en_rango(palabras_modelo, start, end, margen=0.20):
    seleccionadas = []
    left = max(float(start) - margen, 0.0)
    right = float(end) + margen
    for palabra in palabras_modelo:
        p_start = float(palabra.get("start", 0.0))
        p_end = float(palabra.get("end", p_start))
        centro = (p_start + p_end) / 2
        if left <= centro <= right or (p_start < right and p_end > left):
            seleccionadas.append(str(palabra.get("word", "")).strip())
    return " ".join(tok for tok in seleccionadas if tok).strip()


def unir_intervalos(intervalos, gap=0.75):
    intervalos = sorted(intervalos, key=lambda x: (x[0], x[1]))
    unidos = []
    for start, end in intervalos:
        if not unidos or start > unidos[-1][1] + gap:
            unidos.append([float(start), float(end)])
        else:
            unidos[-1][1] = max(unidos[-1][1], float(end))
    return unidos


# Se alinean todos los pares, pero esto queda como paso interno.
# La tabla final muestra solo casos donde los tres modelos difieren.
diferencias_por_par = []
for modelo_a, modelo_b in combinations(MODELOS, 2):
    diferencias = alinear_diferencias(
        palabras[modelo_a],
        palabras[modelo_b],
        contexto=5,
    )
    for diferencia in diferencias:
        diferencias_por_par.append({
            "start": float(diferencia["start"]),
            "end": float(diferencia["end"]),
            "comparacion": f"{modelo_a}_vs_{modelo_b}",
        })

intervalos_diff = [
    (d["start"], d["end"])
    for d in diferencias_por_par
    if d["end"] > d["start"]
]

filas_tres_modelos = []
for start, end in unir_intervalos(intervalos_diff):
    transcripciones_reales = {
        modelo: texto_modelo_en_rango(palabras[modelo], start, end)
        for modelo in MODELOS
    }
    transcripciones_norm = {
        modelo: normalizar_texto(texto)
        for modelo, texto in transcripciones_reales.items()
    }

    if all(transcripciones_norm.values()) and len(set(transcripciones_norm.values())) == len(MODELOS):
        filas_tres_modelos.append({
            "diff_id": len(filas_tres_modelos) + 1,
            "start": round(start, 3),
            "end": round(end, 3),
            "transcripcion_real_turbo": transcripciones_reales["turbo"],
            "transcripcion_real_small": transcripciones_reales["small"],
            "transcripcion_real_large": transcripciones_reales["large"],
            "norm_turbo": transcripciones_norm["turbo"],
            "norm_small": transcripciones_norm["small"],
            "norm_large": transcripciones_norm["large"],
        })

clips_dir = WORKDIR / f"clips_transcripciones_reales_margen_{int(CLIP_MARGIN_SECONDS * 1000):04d}ms"
filas_tres_modelos = exportar_clips_diferencias(
    video_path,
    filas_tres_modelos,
    output_dir=clips_dir,
    margen=CLIP_MARGIN_SECONDS,
    max_clips=len(filas_tres_modelos),
)

df_tres = pd.DataFrame(filas_tres_modelos)
if not df_tres.empty and "review_index" not in df_tres.columns:
    df_tres.insert(0, "review_index", range(len(df_tres)))

df_tres.to_csv(WORKDIR / "diferencias_tres_modelos.csv", index=False)
print(f"Casos donde difieren los tres modelos: {len(df_tres)}")

cols = [
    "review_index",
    "start",
    "end",
    "transcripcion_real_turbo",
    "transcripcion_real_small",
    "transcripcion_real_large",
    "clip_path",
]
display(df_tres[cols] if not df_tres.empty else df_tres)


Casos donde difieren los tres modelos: 80


,review_index,start,end,transcripcion_real_turbo,transcripcion_real_small,transcripcion_real_large,clip_path
0,0,58.46,59.28,"espanto, qué raci.",qué Racing,"espanto, qué Racing.",C:\Users\bianc\NLP Natural Language Processing...
1,1,88.64,89.56,como esta es entrenando,como estas,como esta...,C:\Users\bianc\NLP Natural Language Processing...
2,2,97.64,98.50,cualquier raci. Racingista.,cualquier racingista,cualquier racinguista.,C:\Users\bianc\NLP Natural Language Processing...
3,3,105.56,105.98,diferencia a un equipo,diferencia sobre equipo,diferencia a un equipo de,C:\Users\bianc\NLP Natural Language Processing...
4,4,110.80,111.40,que me duela por,que me duele por,que me duela...,C:\Users\bianc\NLP Natural Language Processing...
...,...,...,...,...,...,...,...
75,75,1024.56,1026.22,"elegir Milito, Saja, la",elegir militos a la secretaría,"elegir Milito, Saja, la Secretaría",C:\Users\bianc\NLP Natural Language Processing...
76,76,1029.12,1034.00,"Que se ocupen ellos. No, pero... Sí, que se oc...","que se ocupe en ellos si, que se ocupe en ello...","Que se ocupen ellos. ¡No! Pero... Sí, que se o...",C:\Users\bianc\NLP Natural Language Processing...
77,77,1041.04,1042.18,"Acá no hay AFA, no hay nada.",no hay aza no hay nada en lo,"no hay aza, no hay nada, eh.",C:\Users\bianc\NLP Natural Language Processing...
78,78,1045.94,1047.12,Acá no hay nada.,hoy perdiste,"Acá no hay nada, eh.",C:\Users\bianc\NLP Natural Language Processing...


## 5. Revisar caso por caso

El texto mostrado es la transcripcion real del video completo en ese intervalo. El clip es solo una ayuda visual para escuchar/ver ese momento.


In [5]:
def ruta_para_notebook(path):
    path = Path(path).resolve()
    rel = os.path.relpath(path, NOTEBOOK_DIR).replace("\\", "/")
    return quote(rel, safe="/:")


if df_tres.empty:
    print("No hay casos donde turbo, small y large tengan tres textos distintos.")
else:
    filas = df_tres if MAX_CASOS_A_MOSTRAR is None else df_tres.head(MAX_CASOS_A_MOSTRAR)
    for _, row in filas.iterrows():
        display(Markdown(
            f"### Caso {int(row['review_index'])} | "
            f"transcripcion original {row['start']}s-{row['end']}s"
        ))

        clip_src = html.escape(ruta_para_notebook(row["clip_path"]), quote=True)
        clip_abs = html.escape(str(Path(row["clip_path"]).resolve()))
        display(HTML(f'''
            <video controls preload="metadata" style="width:min(900px,100%);max-height:70vh;background:#000;" src="{clip_src}"></video>
            <div style="margin:6px 0 12px 0;font-size:13px;">
                <a href="{clip_src}" target="_blank" rel="noopener">Abrir clip en otra pestana</a><br>
                <code>{clip_abs}</code>
            </div>
        '''))

        print("turbo:", row["transcripcion_real_turbo"])
        print("small:", row["transcripcion_real_small"])
        print("large:", row["transcripcion_real_large"])


### Caso 0 | transcripcion original 58.46s-59.28s

turbo: espanto, qué raci.
small: qué Racing
large: espanto, qué Racing.


### Caso 1 | transcripcion original 88.64s-89.56s

turbo: como esta es entrenando
small: como estas
large: como esta...


### Caso 2 | transcripcion original 97.64s-98.5s

...

turbo: Acá no hay AFA, no hay nada.
small: no hay aza no hay nada en lo
large: no hay aza, no hay nada, eh.


### Caso 78 | transcripcion original 1045.94s-1047.12s

turbo: Acá no hay nada.
small: hoy perdiste
large: Acá no hay nada, eh.


### Caso 79 | transcripcion original 1073.76s-1075.76s

turbo: Y bueno. Y bueno. Se viene,
small: digo, si viene no
large: Y bueno, se viene, no


In [6]:
# Opcional: guardar una decision manual para un caso puntual.
REVIEW_INDEX = 0
MODELO_CORRECTO = ""  # opciones: "turbo", "small", "large", "ninguno", "empate"
NOTA = ""

revision_path = WORKDIR / "revision_manual_tres_modelos.csv"

if df_tres.empty:
    print("No hay casos de tres modelos para guardar.")
else:
    if revision_path.exists():
        revision_tres = pd.read_csv(revision_path)
    else:
        revision_tres = df_tres.copy()
        revision_tres["modelo_correcto"] = ""
        revision_tres["nota"] = ""

    if MODELO_CORRECTO:
        mask = revision_tres["review_index"].eq(REVIEW_INDEX)
        revision_tres.loc[mask, "modelo_correcto"] = MODELO_CORRECTO
        revision_tres.loc[mask, "nota"] = NOTA
        revision_tres.to_csv(revision_path, index=False)
        display(revision_tres.loc[mask, ["review_index", "transcripcion_real_turbo", "transcripcion_real_small", "transcripcion_real_large", "modelo_correcto", "nota"]])
        print(f"Guardado en: {revision_path}")
    else:
        completadas = revision_tres["modelo_correcto"].fillna("").ne("").sum()
        print(f"Completa MODELO_CORRECTO para guardar. Revisadas: {completadas} / {len(revision_tres)}")
        display(revision_tres[["review_index", "transcripcion_real_turbo", "transcripcion_real_small", "transcripcion_real_large", "modelo_correcto", "nota"]])


Completa MODELO_CORRECTO para guardar. Revisadas: 0 / 80


,review_index,transcripcion_real_turbo,transcripcion_real_small,transcripcion_real_large,modelo_correcto,nota
0,0,"espanto, qué raci.",qué Racing,"espanto, qué Racing.",,
1,1,como esta es entrenando,como estas,como esta...,,
2,2,cualquier raci. Racingista.,cualquier racingista,cualquier racinguista.,,
3,3,diferencia a un equipo,diferencia sobre equipo,diferencia a un equipo de,,
4,4,que me duela por,que me duele por,que me duela...,,
...,...,...,...,...,...,...
75,75,"elegir Milito, Saja, la",elegir militos a la secretaría,"elegir Milito, Saja, la Secretaría",,
76,76,"Que se ocupen ellos. No, pero... Sí, que se oc...","que se ocupe en ellos si, que se ocupe en ello...","Que se ocupen ellos. ¡No! Pero... Sí, que se o...",,
77,77,"Acá no hay AFA, no hay nada.",no hay aza no hay nada en lo,"no hay aza, no hay nada, eh.",,
78,78,Acá no hay nada.,hoy perdiste,"Acá no hay nada, eh.",,
